In [1]:
# ============================================================
# InternVL3-8B question-aware caching
# Same 13-cell structure as the Qwen2.5-VL notebook.
# Settings: GPU T4 x2, Internet ON.
# Attach inputs: vlm-questions (version 2), amber-hallucination, coco2014.
# ============================================================

!pip install -q -U transformers accelerate requests

import os
import json
import glob
import random
import time
import subprocess
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("torch:", torch.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 72.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32

In [2]:
MODEL_NAME = "OpenGVLab/InternVL3-8B-hf"   # the native-transformers checkpoint (same id the paper used)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

# InternVL splits each image into 448x448 tiles (256 image tokens per tile), so the
# image-token count varies per image. Leave None unless cell 10 runs out of memory;
# then set e.g. 6 to cap the tile count.
MAX_TILES = None
if MAX_TILES is not None:
    assert hasattr(processor.image_processor, "max_patches"), "no max_patches on image_processor -- see cell 2 output"
    processor.image_processor.max_patches = MAX_TILES

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

print("model loaded:", MODEL_NAME, "-", type(model).__name__)


processor_config.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/481 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/781 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

model loaded: OpenGVLab/InternVL3-8B-hf - InternVLForConditionalGeneration


In [3]:
# ============================================================
# DIAGNOSTIC -- read the output. Changes nothing.
# Cell 3 auto-resolves the paths below, and prints what it picked.
# ============================================================

print("=" * 70)
print("MODEL STRUCTURE")
print("=" * 70)
print(model)

def _try(root, path):
    obj = root
    try:
        for part in path.split("."):
            obj = getattr(obj, part)
        return obj
    except AttributeError:
        return None

print()
print("=" * 70)
print("CANDIDATE PATHS (on `model`)")
print("=" * 70)
for path in ["model.language_model", "language_model", "model.language_model.layers",
             "model.language_model.norm", "language_model.model.layers", "language_model.model.norm",
             "model.vision_tower", "vision_tower", "model.multi_modal_projector",
             "multi_modal_projector", "lm_head"]:
    obj = _try(model, path)
    print(f"{path:35s} -> {'FOUND ' + type(obj).__name__ if obj is not None else 'not found'}")

print()
print("=" * 70)
print("CONFIG FIELDS")
print("=" * 70)
_t = getattr(model.config, "text_config", None)
print("config.image_token_id =", getattr(model.config, "image_token_id", "NOT FOUND"))
print("config.image_seq_length =", getattr(model.config, "image_seq_length", "NOT FOUND"))
print("config.downsample_ratio =", getattr(model.config, "downsample_ratio", "NOT FOUND"))
print("text_config.hidden_size =", getattr(_t, "hidden_size", "NOT FOUND"))
print("text_config.num_hidden_layers =", getattr(_t, "num_hidden_layers", "NOT FOUND"))
print("processor.image_token =", getattr(processor, "image_token", "NOT FOUND"))
print("processor.image_token_id =", getattr(processor, "image_token_id", "NOT FOUND"))

print()
print("=" * 70)
print("IMAGE PROCESSOR SETTINGS (tiling)")
print("=" * 70)
print(processor.image_processor)

print()
print("=" * 70)
print("FULL CONFIG DUMP")
print("=" * 70)
print(model.config)


MODEL STRUCTURE
InternVLForConditionalGeneration(
  (model): InternVLModel(
    (vision_tower): InternVLVisionModel(
      (embeddings): InternVLVisionEmbeddings(
        (patch_embeddings): InternVLVisionPatchEmbeddings(
          (projection): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
        )
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (encoder): InternVLVisionEncoder(
        (layer): ModuleList(
          (0-23): 24 x InternVLVisionLayer(
            (attention): InternVLVisionAttention(
              (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (projection_layer): Linear(in_features=1024, out_features=1024, bias=True)
              (projection_dropout): Identity()
              (q_norm): Identity()
              (k_norm): Identity()
            )
            

In [4]:
# Auto-resolve module paths (layouts differ across transformers versions), print
# what was picked, and fail loudly if nothing matches.

def _resolve(root, candidates, what):
    for path in candidates:
        obj = root
        try:
            for part in path.split("."):
                obj = getattr(obj, part)
            print(f"{what:22s} <- model.{path}")
            return obj
        except AttributeError:
            continue
    raise AttributeError(f"could not find {what}; tried {candidates}. Look at cell 2's output and add the right path.")

layers = _resolve(model, ["model.language_model.layers", "language_model.model.layers", "language_model.layers"], "decoder layers")
ln_f = _resolve(model, ["model.language_model.norm", "language_model.model.norm", "language_model.norm"], "final norm (ln_f)")
projector = _resolve(model, ["model.multi_modal_projector", "multi_modal_projector"], "projector (hook target)")
lm_head = model.lm_head

n_layer = len(layers)
hidden_dim = (
    model.config.text_config.hidden_size
    if hasattr(model.config, "text_config")
    else model.config.hidden_size
)

image_token_id = getattr(model.config, "image_token_id", None)
if image_token_id is None:
    image_token_id = processor.image_token_id
assert image_token_id is not None, "no image_token_id found -- see cell 2 output"

print(f"n_layer = {n_layer}")
print(f"hidden_dim = {hidden_dim}")
print(f"image_token_id = {image_token_id}")

# Hook on the projector: its output is the image features already in the LM's hidden
# dimension (same idea as the post-merger hook for Qwen). Shape is (tiles, tokens, hidden)
# or flattened -- pooling in cell 9 handles either.
_encoder_cache = {}

def _vision_hook(module, inputs, output):
    if isinstance(output, tuple):
        out = output[0]
    elif hasattr(output, "last_hidden_state"):
        out = output.last_hidden_state
    else:
        out = output
    _encoder_cache["patch_embeddings"] = out.detach().cpu()

_hook_handle = projector.register_forward_hook(_vision_hook)
print("hook registered on the projector")


decoder layers         <- model.model.language_model.layers
final norm (ln_f)      <- model.model.language_model.norm
projector (hook target) <- model.model.multi_modal_projector
n_layer = 28
hidden_dim = 3584
image_token_id = 151667
hook registered on the projector


In [5]:
def find_file(base, filename):
    for root, _, files in os.walk(base):
        if filename in files:
            return os.path.join(root, filename)
    return None

COCO_ROOT = "/kaggle/input/datasets/nadaibrahim/coco2014"

_known_coco_path = os.path.join(COCO_ROOT, "val2014", "val2014")
if os.path.isdir(_known_coco_path):
    img_dir = _known_coco_path
else:
    sample_path = find_file(COCO_ROOT, "COCO_val2014_000000000139.jpg")
    assert sample_path is not None, f"couldn't find a sample COCO file under {COCO_ROOT} -- check COCO_ROOT"
    img_dir = os.path.dirname(sample_path)
print("COCO images live in:", img_dir)

AMBER_IMAGE_DIR = "/kaggle/input/datasets/ponishbhatia2/amber-hallucination/image"


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


COCO images live in: /kaggle/input/datasets/nadaibrahim/coco2014/val2014/val2014


In [6]:
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]

def find_vg_image_path(image_id: str) -> str:
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None


def resolve_image_path(q: dict) -> str:
    qid = q["question_id"]
    if qid.startswith("pope_"):
        return os.path.join(img_dir, f"{q['image_id']}.jpg")
    elif qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    elif qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None


In [7]:
HALLUC_TYPE = "relation"   # switched from attribute -- this run triggers the Visual Genome download in cell 7
USERNAME = "ponishbhatia2"   # corrected -- was ponishbhatia

# NOTE: confirm this dataset-name convention with Rohan before the first upload --
# later cross-model notebooks look for the exact name.
MODEL_TAG = "internvl3"
dataset_slug = f"question-aware-cache-{MODEL_TAG}-{HALLUC_TYPE}"

QA_CACHE_DIR = f"/kaggle/working/cache/question_aware_{MODEL_TAG}_{HALLUC_TYPE}"
os.makedirs(QA_CACHE_DIR, exist_ok=True)
print(f"HALLUC_TYPE  = {HALLUC_TYPE!r}")
print(f"dataset slug = {USERNAME}/{dataset_slug}")
print(f"output dir   = {QA_CACHE_DIR}")


HALLUC_TYPE  = 'relation'
dataset slug = ponishbhatia2/question-aware-cache-internvl3-relation
output dir   = /kaggle/working/cache/question_aware_internvl3_relation


In [8]:
if HALLUC_TYPE == "relation":
    !wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
    !wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
    !unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
    !unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/
    for zpath in ["/kaggle/working/VG_100K.zip", "/kaggle/working/VG_100K_2.zip"]:
        if os.path.exists(zpath):
            os.remove(zpath)
    print("Visual Genome downloaded, extracted, zips removed")
else:
    print(f"HALLUC_TYPE={HALLUC_TYPE!r} -- Visual Genome not needed, skipping download")


--2026-09-29 20:16:58--  https://cs.stanford.edu/people/rak248/VG_100K/images.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9730308001 (9.1G) [application/zip]
Saving to: ‘/kaggle/working/VG_100K.zip’

/kaggle/working/VG_ 100%[===================>]   9.06G  17.0MB/s    in 31m 56s 

2026-09-29 20:48:54 (4.84 MB/s) - ‘/kaggle/working/VG_100K.zip’ saved [9730308001/9730308001]

--2026-09-29 20:48:55--  https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5471658058 (5.1G) [application/zip]
Saving to: ‘/kaggle/working/VG_100K_2.zip’

/kaggle/working/VG_ 100%[===================>]   5.10G  9.38MB/s    in 13m 15s 

2026-09-29 21:02:11 (6.56 MB

In [9]:
QUESTIONS_PATH = "/kaggle/input/datasets/ponishbhatia2/vlm-questions/all_questions.json"
with open(QUESTIONS_PATH) as f:
    all_questions = json.load(f)

assert len(all_questions) == 17492, "attach vlm-questions version 2"

questions_subset = [
    q for q in all_questions
    if q["hallucination_type"] == HALLUC_TYPE and q["answer_format"] == "yes_no"
]
print(f"{len(questions_subset)} {HALLUC_TYPE} yes/no questions selected")


6506 relation yes/no questions selected


In [10]:
def get_empty_ids(image):
    """
    InternVL tiles each image (tile count depends on the image's shape), so the number
    of image tokens varies per image. The empty template must therefore be rebuilt per
    image size: a blank image of the same size tiles the same way.
    """
    blank = Image.new("RGB", image.size)
    return build_inputs(blank, text="")["input_ids"][0].tolist()


def question_span(ids, empty_ids):
    n = min(len(ids), len(empty_ids))
    start = next(i for i in range(n) if ids[i] != empty_ids[i])
    tail = next(i for i in range(n) if ids[-1 - i] != empty_ids[-1 - i])
    return start, len(ids) - tail


@torch.no_grad()
def cache_question_aware(image_path, image_id, question_id, question_text):
    image = load_image(image_path)
    inputs = build_inputs(image, text=question_text)
    inputs.pop("token_type_ids", None)   # some processors return it; the model forward doesn't take it
    ids = inputs["input_ids"][0].tolist()

    empty_ids = get_empty_ids(image)
    q_start, q_end = question_span(ids, empty_ids)

    _encoder_cache.clear()
    outputs = model(**inputs, output_hidden_states=True)
    assert "patch_embeddings" in _encoder_cache, "vision hook didn't fire -- check the hook target in cell 3"
    last_logits = outputs.logits[0, -1]

    image_positions = (inputs["input_ids"][0] == image_token_id).nonzero(as_tuple=True)[0]
    assert image_positions.numel() > 0, "no image tokens found -- check image_token_id in cell 3"

    p1_hidden = torch.stack([layer[0, -1, :].half().cpu() for layer in outputs.hidden_states])
    p2_hidden = torch.stack([layer[0, q_start:q_end, :].mean(dim=0).half().cpu() for layer in outputs.hidden_states])
    pooled_image_hidden = torch.stack([layer[0, image_positions, :].mean(dim=0).half().cpu() for layer in outputs.hidden_states])

    enc = _encoder_cache["patch_embeddings"]
    pooled_encoder_embedding = enc.reshape(-1, enc.shape[-1]).float().mean(dim=0).half()

    return {
        "image_id": image_id, "question_id": question_id, "question_text": question_text,
        "answer_text": processor.tokenizer.decode(last_logits.argmax().item()).strip(),
        "span_text": processor.tokenizer.decode(ids[q_start:q_end]),
        "n_image_tokens": int(image_positions.numel()),
        "p1_hidden": p1_hidden,
        "p2_hidden": p2_hidden,
        "pooled_image_hidden": pooled_image_hidden,
        "pooled_encoder_embedding": pooled_encoder_embedding,
        "answer_logits": last_logits.half().cpu(),
    }


In [11]:
# ============================================================
# VALIDATION CHECKPOINT -- read all of it before cell 11.
# ============================================================

random.seed(42)
val_subset = random.sample(questions_subset, min(50, len(questions_subset)))

t0 = time.time()
val_entries, val_qs = [], []
for q in val_subset:
    image_path = resolve_image_path(q)
    if image_path is None or not os.path.exists(image_path):
        continue
    val_entries.append(cache_question_aware(image_path, q["image_id"], q["question_id"], q["question_text"]))
    val_qs.append(q)
sec_per_question = (time.time() - t0) / len(val_entries)

print(f"{len(val_entries)} questions validated, {sec_per_question:.2f} s/question")
print()

# --- hidden_states count and NaN/inf check (fp16 can overflow) ---
n_states = val_entries[0]["p1_hidden"].shape[0]
print(f"hidden_states entries per question: {n_states} (expected n_layer + 1 = {n_layer + 1})")
assert n_states == n_layer + 1, "hidden_states count mismatch -- check cell 3's layer path"
bad = sum(
    (not torch.isfinite(e["p1_hidden"].float()).all()) or (not torch.isfinite(e["answer_logits"].float()).all())
    for e in val_entries
)
print(f"entries with NaN/inf: {bad}/{len(val_entries)}")
if bad:
    print(">>> fp16 overflow -- stop here and tell Ponish/Rohan before running cell 11.")
print()

# --- normalization convention: exactly one of these should be near zero ---
raw_err = max(
    (lm_head(e["p1_hidden"][-1].to(lm_head.weight.device)).float().cpu() - e["answer_logits"].float()).abs().max().item()
    for e in val_entries
)
normed_err = max(
    (lm_head(ln_f(e["p1_hidden"][-1].to(lm_head.weight.device))).float().cpu() - e["answer_logits"].float()).abs().max().item()
    for e in val_entries
)
print("--- normalization convention check ---")
print(f"lm_head(p1_hidden[-1]) directly vs real logits, max abs err:    {raw_err:.4f}")
print(f"lm_head(ln_f(p1_hidden[-1])) vs real logits, max abs err:       {normed_err:.4f}")
if raw_err < 0.1 and normed_err >= 0.1:
    print(">>> convention: hidden_states[-1] is ALREADY post-final-norm. No extra ln_f needed at the last layer.")
elif normed_err < 0.1 and raw_err >= 0.1:
    print(">>> convention: hidden_states[-1] is PRE-final-norm. Downstream lens code MUST apply ln_f first.")
else:
    print(">>> NEITHER (or both) near zero -- something upstream is wrong. Recheck cell 3's paths.")
print()

layer0_identical = all(torch.equal(e["p1_hidden"][0], val_entries[0]["p1_hidden"][0]) for e in val_entries)
print(f"layer-0 p1 identical across questions: {layer0_identical}")

spans_match = sum(e["span_text"].strip() == q["question_text"].strip() for e, q in zip(val_entries, val_qs))
print(f"question span decodes to the question: {spans_match}/{len(val_entries)}")

n_img = [e["n_image_tokens"] for e in val_entries]
print(f"image token count range: {min(n_img)} - {max(n_img)}  (expect multiples of the per-tile count, 256)")
print()

print("--- sample answers (read these yourself) ---")
for e in val_entries[:15]:
    print(f"  {e['question_id']:20s} {e['question_text'][:50]:50s} -> {e['answer_text']!r}")

print()
print("Don't run cell 11 until all of the above looks right.")


50 questions validated, 4.42 s/question

hidden_states entries per question: 29 (expected n_layer + 1 = 29)
entries with NaN/inf: 0/50

--- normalization convention check ---
lm_head(p1_hidden[-1]) directly vs real logits, max abs err:    0.0156
lm_head(ln_f(p1_hidden[-1])) vs real logits, max abs err:       23.8438
>>> convention: hidden_states[-1] is ALREADY post-final-norm. No extra ln_f needed at the last layer.

layer-0 p1 identical across questions: True
question span decodes to the question: 50/50
image token count range: 256 - 3328  (expect multiples of the per-tile count, 256)

--- sample answers (read these yourself) ---
  reefknot_2349891_0   Is the man taking picture in this photo? Please an -> 'yes'
  amber_14469          Is there direct contact between the bird and beach -> 'Yes'
  amber_13761          Is there direct contact between the child and gras -> 'Yes'
  reefknot_2415389_0   Is the man teaching boy in this photo? Please answ -> 'yes'
  reefknot_2332110_1   Is the

In [12]:
times = []
for q in val_subset[:10]:
    t0 = time.time()
    cache_question_aware(resolve_image_path(q), q["image_id"], q["question_id"], q["question_text"])
    times.append(time.time() - t0)
print(sorted(times))

[0.4318833351135254, 2.945695638656616, 2.9724411964416504, 2.979196786880493, 6.953845500946045, 6.999740839004517, 7.0016279220581055, 7.015734672546387, 7.0229668617248535, 7.037623167037964]


In [13]:
UPLOAD_EVERY = 250
BATCH_SAVE_SIZE = UPLOAD_EVERY   # every 250 cached questions = 1 new file = 1 upload
FORCE_FRESH = False              # True only if you've checked your Kaggle profile and the dataset really isn't there

assert USERNAME and not USERNAME.startswith("REPLACE"), "set USERNAME in cell 6"


def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    ok = r.returncode == 0   # trust the exit code; scanning output text for "error" gave false negatives
    return ok, out


# ---- 1. restore anything already saved in the Kaggle dataset ----------------
ok, out = _run(["kaggle", "datasets", "download", "-d", f"{USERNAME}/{dataset_slug}",
                "-p", QA_CACHE_DIR, "--unzip"])
if ok:
    print("restored existing cache files from the Kaggle dataset")
else:
    low = out.lower()
    looks_missing = any(s in low for s in ("404", "403", "not found", "does not exist", "forbidden"))
    if not looks_missing and not FORCE_FRESH:
        raise RuntimeError("download failed for an unrelated reason, refusing to start fresh:\n" + out)
    if not FORCE_FRESH:
        # Kaggle answers 403 for both "doesn't exist" and "no access": check your own dataset list
        lok, lout = _run(["kaggle", "datasets", "list", "-m", "-s", dataset_slug, "-v"])
        if not lok:
            raise RuntimeError("can't verify your Kaggle login/credentials. If you're sure nothing is "
                               "uploaded yet, set FORCE_FRESH = True and re-run:\n" + lout)
        if f"{USERNAME}/{dataset_slug}" in lout:
            raise RuntimeError("dataset exists but the download was forbidden:\n" + out)
    print("no existing dataset on Kaggle yet, starting fresh")

with open(os.path.join(QA_CACHE_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({"title": dataset_slug, "id": f"{USERNAME}/{dataset_slug}",
               "licenses": [{"name": "CC0-1.0"}]}, f)

dataset_exists = ok   # True if we just restored one, so the next upload is a "version"


def upload_to_kaggle(msg):
    """Upload the whole folder. A failure warns and continues; the files go up with the next upload."""
    global dataset_exists
    t0 = time.time()
    create_cmd = ["kaggle", "datasets", "create", "-p", QA_CACHE_DIR]
    version_cmd = ["kaggle", "datasets", "version", "-p", QA_CACHE_DIR, "-m", msg]
    if not dataset_exists:
        ok_, out_ = _run(create_cmd)
        if not ok_:
            ok_, out2 = _run(version_cmd)
            out_ += "\n" + out2
    else:
        ok_, out_ = _run(version_cmd)
    if ok_:
        dataset_exists = True
        print(f"    [upload ok in {time.time()-t0:.0f}s] {msg}")
    else:
        print(f"    [UPLOAD FAILED, continuing] {out_[-400:]}")
    return ok_


# ---- 2. resume: skip every question already saved ---------------------------
existing_batches = glob.glob(os.path.join(QA_CACHE_DIR, f"{HALLUC_TYPE}_batch_*.pt"))
already_done_ids = set()
for fpath in existing_batches:
    already_done_ids.update(e["question_id"] for e in torch.load(fpath))
print(f"resuming: {len(already_done_ids)} questions already saved in {len(existing_batches)} files")

batch_index = len(existing_batches)
batch_entries, skipped, span_mismatch = [], 0, 0
t_start = time.time()

# ---- 3. main loop ------------------------------------------------------------
for i, q in enumerate(questions_subset):
    if q["question_id"] in already_done_ids:
        continue

    image_path = resolve_image_path(q)
    if image_path is None or not os.path.exists(image_path):
        skipped += 1
        continue

    entry = cache_question_aware(image_path, q["image_id"], q["question_id"], q["question_text"])
    entry.update({k: q[k] for k in ("ground_truth_answer", "answer_format", "hallucination_type")})
    span_mismatch += entry["span_text"].strip() != q["question_text"].strip()
    batch_entries.append(entry)

    if len(batch_entries) >= BATCH_SAVE_SIZE:
        torch.save(batch_entries, os.path.join(QA_CACHE_DIR, f"{HALLUC_TYPE}_batch_{batch_index}.pt"))
        batch_index += 1
        batch_entries = []
        upload_to_kaggle(f"{HALLUC_TYPE}: {batch_index} batches, through question {i+1}/{len(questions_subset)}")

    if i % 10 == 0 and i > 0:
        elapsed = time.time() - t_start
        avg = elapsed / (i + 1)
        remaining = avg * (len(questions_subset) - i - 1)
        print(f"[{i+1}/{len(questions_subset)}] {q['question_id']}: {entry['answer_text']!r} "
              f"({avg:.2f}s/question avg, ETA {remaining/60:.1f} min, skipped: {skipped}, "
              f"span mismatches: {span_mismatch}, img_tokens: {entry['n_image_tokens']})")

if batch_entries:
    torch.save(batch_entries, os.path.join(QA_CACHE_DIR, f"{HALLUC_TYPE}_batch_{batch_index}.pt"))
    batch_index += 1
upload_to_kaggle(f"{HALLUC_TYPE}: FINAL, {batch_index} batches")

print(f"done -- {batch_index} files, {skipped} skipped, {span_mismatch} span mismatches, "
      f"{(time.time()-t_start)/60:.1f} min this session")


no existing dataset on Kaggle yet, starting fresh
resuming: 0 questions already saved in 0 files
[11/6506] amber_13567: 'Yes' (6.31s/question avg, ETA 683.5 min, skipped: 0, span mismatches: 0, img_tokens: 1792)
[21/6506] amber_13577: 'Yes' (5.05s/question avg, ETA 545.6 min, skipped: 0, span mismatches: 0, img_tokens: 256)
[31/6506] amber_13587: 'No' (5.21s/question avg, ETA 562.7 min, skipped: 0, span mismatches: 0, img_tokens: 3328)
[41/6506] amber_13597: 'No' (5.14s/question avg, ETA 553.9 min, skipped: 0, span mismatches: 0, img_tokens: 3328)
[51/6506] amber_13607: 'No' (5.20s/question avg, ETA 559.4 min, skipped: 0, span mismatches: 0, img_tokens: 3328)
[61/6506] amber_13617: 'Yes' (4.88s/question avg, ETA 524.6 min, skipped: 0, span mismatches: 0, img_tokens: 3328)
[71/6506] amber_13627: 'No' (4.85s/question avg, ETA 519.7 min, skipped: 0, span mismatches: 0, img_tokens: 1792)
[81/6506] amber_13637: 'Yes' (4.92s/question avg, ETA 526.6 min, skipped: 0, span mismatches: 0, img_to

In [14]:
if skipped:
    print(f"WARNING: {skipped} questions were skipped this session (missing images), so the dataset may be incomplete")

ok, out = _run(["kaggle", "datasets", "files", f"{USERNAME}/{dataset_slug}"])
print(out)
print()
print(f"published: {USERNAME}/{dataset_slug}")
print("For Rohan: send back this dataset link, the full printed output of cells 2, 3 and 10")
print("(the module paths and the normalization-convention result matter most),")
print("and total GPU time used for this run.")


Next Page Token = CfDJ8LXGgraiXMNClPzcySXY5U4H5U8nLySvoa67oYofR47Wygh-z5vyAWgRQp14L2Ze8fwYtmBDuBlzavFzRcOMgiZns9R2GHxKoTYtsfBGHXb67TgbbJNIpF2Qnt5tyolDVczm_PdBMxmSM1CpWQ
name                       size  creationDate                
--------------------  ---------  --------------------------  
relation_batch_0.pt   233934237  2026-09-30 06:04:01.540000  
relation_batch_1.pt   233934493  2026-09-30 06:04:01.836000  
relation_batch_10.pt  233947589  2026-09-30 06:04:00.335000  
relation_batch_11.pt  233947269  2026-09-30 06:04:02.042000  
relation_batch_12.pt  233947909  2026-09-30 06:04:01.628000  
relation_batch_13.pt  233947717  2026-09-30 06:04:03.464000  
relation_batch_14.pt  233948037  2026-09-30 06:04:01.576000  
relation_batch_15.pt  233947461  2026-09-30 06:04:01.855000  
relation_batch_16.pt  233947525  2026-09-30 06:04:01.887000  
relation_batch_17.pt  233948165  2026-09-30 06:04:01.605000  
relation_batch_18.pt  233947973  2026-09-30 06:04:02.636000  
relation_batch_19.pt  233